In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from db import query

# Style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = {
    'control':    '#2ecc71',
    'spurious':   '#e74c3c',
    'text':       '#3498db',
    'embeddings': '#3498db',
    'lda':        '#9b59b6',
    'sentiment':  '#f39c12',
}

# Load all results
summary = query("SELECT * FROM detection_summary")
bonf    = query("SELECT * FROM bonferroni_fdr_results")
boot    = query("SELECT * FROM bootstrap_results")
wf      = query("SELECT * FROM walkforward_results")
meta    = pd.read_csv('../results/metafeature_descriptors.csv')

os.makedirs('plots', exist_ok=True)

def subgroup(feat):
    if feat == 'sentiment_score': return 'sentiment'
    if str(feat).startswith('lda_topic'): return 'lda'
    if str(feat).startswith('embed_'): return 'embeddings'
    return None

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

methods = summary['method'].tolist()
x = np.arange(len(methods))
width = 0.35

bars_tpr = ax.bar(x - width/2, summary['tpr'], width, label='TPR (Spurious Detected)',
                   color='#e74c3c', alpha=0.85, edgecolor='black', linewidth=0.5)
bars_fpr = ax.bar(x + width/2, summary['fpr'], width, label='FPR (Control Misclassified)',
                   color='#2ecc71', alpha=0.85, edgecolor='black', linewidth=0.5)

# Annotate bars
for bar in bars_tpr:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars_fpr:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.set_ylabel('Rate', fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('RQ1: Detection Method Performance\n(TPR = spurious correctly flagged, FPR = control incorrectly flagged)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='random baseline')

plt.tight_layout()
plt.savefig('plots/rq1_detection_performance.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, group in zip(axes, ['control', 'spurious', 'text']):
    subset = bonf[bonf['feature_group'] == group]['p_value']
    n = len(subset)
    rejected = bonf[bonf['feature_group'] == group]['bonferroni_rejected'].sum()
    ax.hist(subset, bins=30, color=COLORS[group], edgecolor='black',
            linewidth=0.4, alpha=0.85)
    ax.axvline(0.05 / len(bonf), color='red', linestyle='--',
               linewidth=1.2, label=f'Bonferroni \u03b1')
    ax.set_title(f'{group.capitalize()}\n(n={n}, rejected={rejected})',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('p-value', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('RQ1: P-value Distributions by Feature Group',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/rq1_pvalue_distributions.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for group in ['control', 'spurious', 'text']:
    subset = wf[wf['feature_group'] == group]['mean_gen_ratio']
    ax.hist(subset, bins=50, alpha=0.5, label=group.capitalize(),
            color=COLORS[group], edgecolor='none')

ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='ratio = 0')
ax.axvline(0.5, color='gray', linestyle=':', linewidth=1.2, label='threshold = 0.5')
ax.set_xlabel('Mean Generalization Ratio (clipped \u00b110)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('RQ1: Walk-Forward Generalization Ratio Distribution\n(Indistinguishable across groups \u2014 null result)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('plots/rq1_walkforward_distribution.png', dpi=150)
plt.show()

In [ ]:
coef_data = {
    'feature': ['group_embeddings', 'group_lda_topics', 'corr_stability',
                'group_sentiment', 'group_spurious'],
    'coefficient': [1.258467, 0.578985, 0.271630, 0.161814, 0.139477]
}
coef_df = pd.DataFrame(coef_data).sort_values('coefficient', ascending=True)

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_df['coefficient']]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(coef_df['feature'], coef_df['coefficient'],
               color=colors, edgecolor='black', linewidth=0.5, alpha=0.85)

for bar, val in zip(bars, coef_df['coefficient']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (L1 Logistic Regression)', fontsize=11)
ax.set_title('RQ2: Meta-Feature Regression \u2014 Predictors of Spuriousness\n'
             '(Higher = stronger predictor of spurious label)',
             fontsize=13, fontweight='bold')
ax.set_yticklabels([
    'Embeddings Group', 'LDA Topics Group', 'Corr. Stability',
    'Sentiment Group', 'Spurious Group'
], fontsize=10)

plt.tight_layout()
plt.savefig('plots/rq2_meta_coefficients.png', dpi=150)
plt.show()

In [ ]:
# Build sub-modality rejection rates
def get_text_subgroup_rates(df, rejected_col):
    df = df[df['feature_group'] == 'text'].copy()
    df['subgroup'] = df['feature'].apply(subgroup)
    return df.groupby('subgroup')[rejected_col].mean()

rates = pd.DataFrame({
    'Bonferroni': get_text_subgroup_rates(bonf, 'bonferroni_rejected'),
    'FDR':        get_text_subgroup_rates(bonf, 'fdr_rejected'),
    'Bootstrap':  get_text_subgroup_rates(boot, 'bootstrap_rejected'),
    'Walk-Fwd':   get_text_subgroup_rates(wf,   'wf_rejected'),
}).loc[['sentiment', 'lda', 'embeddings']]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(rates.index))
width = 0.2
method_colors = ['#2c3e50', '#7f8c8d', '#e67e22', '#16a085']

for i, (method, color) in enumerate(zip(rates.columns, method_colors)):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, rates[method], width, label=method,
                  color=color, alpha=0.85, edgecolor='black', linewidth=0.4)
    for bar in bars:
        if bar.get_height() > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(['Sentiment', 'LDA Topics', 'Embeddings'], fontsize=12)
ax.set_ylabel('Rejection Rate', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('RQ3: Rejection Rate by Text Sub-Modality and Detection Method',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('plots/rq3_text_submodality.png', dpi=150)
plt.show()

In [ ]:
# Detection summary
print('=' * 60)
print('FINAL DETECTION SUMMARY')
print('=' * 60)
display(summary[['method', 'tpr', 'fpr', 'f1']].round(3))

# Text sub-modality table
print('\nTEXT SUB-MODALITY REJECTION RATES')
print('=' * 60)
display(rates.round(3))

# Meta-feature regression summary
print('\nMETA-FEATURE REGRESSION')
print('=' * 60)
print('CV Accuracy: 0.990 \u00b1 0.009')
print('CV AUC:      0.996 \u00b1 0.005')
print('Classification Report (known labels):')
print('  Genuine:  precision=0.83, recall=0.71, f1=0.77')
print('  Spurious: precision=0.71, recall=0.83, f1=0.77')
print('  Accuracy: 0.77 (n=13)')

# Save summary tables to CSV
os.makedirs('../results', exist_ok=True)
summary.to_csv('../results/FINAL_SUMMARY.csv', index=False)
rates.to_csv('../results/text_submodality_summary.csv')
print('\nSaved FINAL_SUMMARY.csv and text_submodality_summary.csv')